# Exercises — Language Model Probability Math

**Chapter 15, Section 05**

Eight hands-on exercises covering the core mathematics of language model probability:

| # | Topic | Difficulty |
|---|-------|-----------|
| 1 | Chain Rule Decomposition | ⭐ |
| 2 | Perplexity Calculation | ⭐⭐ |
| 3 | Temperature Effect | ⭐⭐ |
| 4 | Top-p (Nucleus) Sampling | ⭐⭐ |
| 5 | Cross-Entropy Gradient | ⭐⭐⭐ |
| 6 | Scaling Law Extrapolation | ⭐⭐⭐ |
| 7 | DPO Loss by Hand | ⭐⭐⭐ |
| 8 | Calibration Check (ECE) | ⭐⭐⭐ |

Each exercise has: **Problem → Scaffold → Solution**

*Reference: [notes.md](./notes.md) and [theory.ipynb](./theory.ipynb)*

In [ ]:
# === Setup ===
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def softmax_stable(z):
    """Numerically stable softmax."""
    z = np.asarray(z, dtype=float)
    z_shifted = z - np.max(z)
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum()

def sigmoid(x):
    """Numerically stable sigmoid."""
    return np.where(x >= 0, 1/(1+np.exp(-x)), np.exp(x)/(1+np.exp(x)))

print("Setup complete ✓")

---
## Exercise 1: Chain Rule Decomposition ⭐

**Problem:** Given a sequence $w_1 w_2 w_3 w_4$ = "the cat sat down", decompose its probability using the chain rule:

$$P(w_1, w_2, w_3, w_4) = P(w_1) \cdot P(w_2|w_1) \cdot P(w_3|w_1,w_2) \cdot P(w_4|w_1,w_2,w_3)$$

You are given these conditional probabilities:
- $P(\text{the}) = 0.15$
- $P(\text{cat} \mid \text{the}) = 0.08$
- $P(\text{sat} \mid \text{the cat}) = 0.12$
- $P(\text{down} \mid \text{the cat sat}) = 0.25$

**Tasks:**
1. Compute $P(\text{the cat sat down})$
2. Compute $\log P(\text{the cat sat down})$
3. Compute per-token log probability
4. Compute perplexity of this sequence
5. If "the dog sat down" has $P(\text{dog}|\text{the})=0.05$ and rest same, which is more likely?

In [ ]:
# === Exercise 1: Scaffold ===

# Given conditional probabilities
p_the = 0.15
p_cat_given_the = 0.08
p_sat_given_the_cat = 0.12
p_down_given_the_cat_sat = 0.25

# Task 1: Compute joint probability P(the cat sat down)
p_sequence = None  # TODO: multiply the conditionals

# Task 2: Compute log probability
log_p_sequence = None  # TODO: sum of log conditionals

# Task 3: Per-token average log probability
avg_log_p = None  # TODO: log_p_sequence / number_of_tokens

# Task 4: Perplexity
perplexity = None  # TODO: exp(-avg_log_p)

# Task 5: Compare with "the dog sat down" (P(dog|the) = 0.05)
p_dog_given_the = 0.05
p_sequence_dog = None  # TODO

print(f"P(the cat sat down)     = {p_sequence}")
print(f"log P(the cat sat down) = {log_p_sequence}")
print(f"Average log P per token = {avg_log_p}")
print(f"Perplexity              = {perplexity}")
print(f"P(the dog sat down)     = {p_sequence_dog}")
print(f"More likely sequence    = ???")

In [ ]:
# === Exercise 1: Solution ===

# Given conditional probabilities
p_the = 0.15
p_cat_given_the = 0.08
p_sat_given_the_cat = 0.12
p_down_given_the_cat_sat = 0.25

# Task 1: Joint probability via chain rule
p_sequence = p_the * p_cat_given_the * p_sat_given_the_cat * p_down_given_the_cat_sat
print(f"P(the cat sat down) = {p_the} × {p_cat_given_the} × {p_sat_given_the_cat} × {p_down_given_the_cat_sat}")
print(f"                    = {p_sequence:.6f}")

# Task 2: Log probability (sum of logs = log of product)
cond_probs = [p_the, p_cat_given_the, p_sat_given_the_cat, p_down_given_the_cat_sat]
log_p_sequence = sum(np.log(p) for p in cond_probs)
print(f"\nlog P = Σ log P(wᵢ|w<ᵢ) = {' + '.join(f'{np.log(p):.4f}' for p in cond_probs)}")
print(f"      = {log_p_sequence:.6f}")
print(f"  Verify: exp(log P) = {np.exp(log_p_sequence):.6f} ≈ {p_sequence:.6f} ✓")

# Task 3: Per-token average
n_tokens = 4
avg_log_p = log_p_sequence / n_tokens
print(f"\nAverage log P per token = {log_p_sequence:.6f} / {n_tokens} = {avg_log_p:.6f}")

# Task 4: Perplexity
perplexity = np.exp(-avg_log_p)
print(f"Perplexity = exp(-avg log P) = exp({-avg_log_p:.6f}) = {perplexity:.2f}")
print(f"  Interpretation: model is as uncertain as choosing uniformly from ~{perplexity:.0f} tokens")

# Task 5: Compare
p_sequence_dog = p_the * p_dog_given_the * p_sat_given_the_cat * p_down_given_the_cat_sat
log_p_dog = np.log(p_the) + np.log(p_dog_given_the) + np.log(p_sat_given_the_cat) + np.log(p_down_given_the_cat_sat)
ppl_dog = np.exp(-log_p_dog / n_tokens)
print(f"\nP(the dog sat down) = {p_sequence_dog:.6f}")
print(f"log P = {log_p_dog:.6f}, PPL = {ppl_dog:.2f}")
print(f"'the cat sat down' PPL={perplexity:.2f} vs 'the dog sat down' PPL={ppl_dog:.2f}")
print(f"More likely: {'cat' if p_sequence > p_sequence_dog else 'dog'} version (lower PPL = more likely)")

---
## Exercise 2: Perplexity Calculation ⭐⭐

**Problem:** A language model assigns the following log-probabilities to a 10-token sequence:

| Position | Token | log P(token | context) |
|----------|-------|----------------------|
| 1 | The | -1.20 |
| 2 | quick | -3.50 |
| 3 | brown | -2.10 |
| 4 | fox | -1.80 |
| 5 | jumps | -2.40 |
| 6 | over | -0.90 |
| 7 | the | -0.50 |
| 8 | lazy | -2.80 |
| 9 | dog | -1.10 |
| 10 | . | -0.80 |

**Tasks:**
1. Compute total log-likelihood
2. Compute per-token cross-entropy $H = -\frac{1}{T}\sum \log P(w_t \mid w_{<t})$
3. Compute perplexity $\text{PPL} = \exp(H)$
4. Compute bits-per-token $\text{BPT} = H / \ln 2$
5. Which tokens does the model find most/least surprising?
6. If we remove token 2 ("quick"), how does perplexity change?

In [ ]:
# === Exercise 2: Scaffold ===

tokens = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog", "."]
log_probs = [-1.20, -3.50, -2.10, -1.80, -2.40, -0.90, -0.50, -2.80, -1.10, -0.80]

# Task 1: Total log-likelihood
total_ll = None  # TODO

# Task 2: Per-token cross-entropy H
H = None  # TODO

# Task 3: Perplexity
ppl = None  # TODO

# Task 4: Bits per token
bpt = None  # TODO

# Task 5: Most/least surprising
most_surprising = None  # TODO: token with most negative log-prob
least_surprising = None  # TODO: token with least negative log-prob

# Task 6: Perplexity without "quick"
# TODO: remove index 1, recompute

print(f"Total log-likelihood: {total_ll}")
print(f"Cross-entropy H:      {H}")
print(f"Perplexity:           {ppl}")
print(f"Bits per token:       {bpt}")
print(f"Most surprising:      {most_surprising}")
print(f"Least surprising:     {least_surprising}")

In [ ]:
# === Exercise 2: Solution ===

tokens = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog", "."]
log_probs = np.array([-1.20, -3.50, -2.10, -1.80, -2.40, -0.90, -0.50, -2.80, -1.10, -0.80])
T = len(tokens)

# Task 1: Total log-likelihood
total_ll = np.sum(log_probs)
print(f"Total log-likelihood = Σ log P = {total_ll:.2f}")

# Task 2: Per-token cross-entropy
H = -total_ll / T
print(f"Cross-entropy H = -({total_ll:.2f})/{T} = {H:.4f} nats/token")

# Task 3: Perplexity
ppl = np.exp(H)
print(f"Perplexity = exp(H) = exp({H:.4f}) = {ppl:.2f}")

# Task 4: Bits per token
bpt = H / np.log(2)
print(f"Bits per token = H/ln(2) = {H:.4f}/{np.log(2):.4f} = {bpt:.4f} bits/token")

# Task 5: Surprisal analysis
print(f"\nPer-token surprisal breakdown:")
print(f"{'Token':<10} {'log P':<10} {'Surprisal (-log P)':<20} {'P'}")
print("-" * 50)
for tok, lp in zip(tokens, log_probs):
    print(f"{tok:<10} {lp:<10.2f} {-lp:<20.2f} {np.exp(lp):.4f}")

most_idx = np.argmin(log_probs)  # most negative = most surprising
least_idx = np.argmax(log_probs)  # least negative = least surprising
print(f"\nMost surprising:  '{tokens[most_idx]}' (log P = {log_probs[most_idx]:.2f}, P = {np.exp(log_probs[most_idx]):.4f})")
print(f"Least surprising: '{tokens[least_idx]}' (log P = {log_probs[least_idx]:.2f}, P = {np.exp(log_probs[least_idx]):.4f})")

# Task 6: Without "quick"
log_probs_no_quick = np.delete(log_probs, 1)
T_new = len(log_probs_no_quick)
H_new = -np.sum(log_probs_no_quick) / T_new
ppl_new = np.exp(H_new)
print(f"\nWithout 'quick': H = {H_new:.4f}, PPL = {ppl_new:.2f}")
print(f"PPL dropped from {ppl:.2f} → {ppl_new:.2f}")
print(f"Insight: removing a surprising token LOWERS perplexity — the model was uncertain about 'quick'")

---
## Exercise 3: Temperature Effect ⭐⭐

**Problem:** Given logits $z = [3.0, 1.0, 0.5, -1.0]$ for vocabulary $[\text{Paris}, \text{London}, \text{Berlin}, \text{banana}]$:

**Tasks:**
1. Compute $\text{softmax}(z/\tau)$ for $\tau \in \{0.1, 0.5, 1.0, 2.0, 5.0\}$
2. For each temperature, compute the **entropy** $H = -\sum p_i \log p_i$
3. Show that $\tau \to 0$ gives argmax (greedy) and $\tau \to \infty$ gives uniform
4. Find the temperature $\tau^*$ that makes $H = 1.0$ nat (binary search)
5. Plot the probability of each token vs temperature

In [ ]:
# === Exercise 3: Scaffold ===

z = np.array([3.0, 1.0, 0.5, -1.0])
vocab = ["Paris", "London", "Berlin", "banana"]
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

# Task 1: Compute softmax(z/τ) for each temperature
for tau in temperatures:
    p = None  # TODO: softmax_stable(z / tau)
    print(f"τ={tau}: {p}")

# Task 2: Compute entropy for each
for tau in temperatures:
    p = None  # TODO
    H = None  # TODO: -sum(p * log(p))
    print(f"τ={tau}: H = {H}")

# Task 3: Show limits
# TODO: very small τ → one-hot; very large τ → uniform

# Task 4: Binary search for τ* where H = 1.0
# TODO

# Task 5: Plot (if matplotlib available)
# TODO

In [ ]:
# === Exercise 3: Solution ===

z = np.array([3.0, 1.0, 0.5, -1.0])
vocab = ["Paris", "London", "Berlin", "banana"]
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

# Task 1 & 2: Softmax and entropy at each temperature
def entropy(p):
    p = p[p > 0]
    return -np.sum(p * np.log(p))

print("=== Temperature Effect on softmax([3.0, 1.0, 0.5, -1.0]) ===\n")
print(f"{'τ':<6} {'P(Paris)':<10} {'P(London)':<11} {'P(Berlin)':<11} {'P(banana)':<11} {'H (nats)':<10} {'PPL':<8}")
print("-" * 70)
for tau in temperatures:
    p = softmax_stable(z / tau)
    H = entropy(p)
    ppl = np.exp(H)
    print(f"{tau:<6.1f} {p[0]:<10.4f} {p[1]:<11.4f} {p[2]:<11.4f} {p[3]:<11.4f} {H:<10.4f} {ppl:<8.2f}")

# Task 3: Limiting behaviour
print("\n=== Limiting Behaviour ===")
p_cold = softmax_stable(z / 0.001)
p_hot = softmax_stable(z / 100.0)
print(f"τ→0:  {p_cold} → one-hot at argmax (Paris)")
print(f"τ→∞:  {p_hot} → uniform [{1/4:.4f}, {1/4:.4f}, {1/4:.4f}, {1/4:.4f}]")
print(f"Max entropy (uniform) = ln(4) = {np.log(4):.4f}")

# Task 4: Binary search for τ* where H = 1.0 nat
target_H = 1.0
lo, hi = 0.01, 20.0
for _ in range(100):
    mid = (lo + hi) / 2
    p = softmax_stable(z / mid)
    H_mid = entropy(p)
    if H_mid < target_H:
        lo = mid
    else:
        hi = mid
tau_star = (lo + hi) / 2
p_star = softmax_stable(z / tau_star)
print(f"\nτ* for H=1.0 nat: τ* = {tau_star:.4f}")
print(f"  softmax(z/τ*) = [{', '.join(f'{x:.4f}' for x in p_star)}]")
print(f"  H = {entropy(p_star):.4f} ≈ 1.0 ✓")

# Task 5: Visualization
if HAS_MPL:
    tau_range = np.linspace(0.05, 5.0, 200)
    probs_all = np.array([softmax_stable(z / t) for t in tau_range])
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    ax = axes[0]
    for i, name in enumerate(vocab):
        ax.plot(tau_range, probs_all[:, i], linewidth=2, label=name)
    ax.axvline(tau_star, color='gray', linestyle='--', alpha=0.5, label=f'τ*={tau_star:.2f}')
    ax.set_xlabel('Temperature τ')
    ax.set_ylabel('Probability')
    ax.set_title('Token Probabilities vs Temperature')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    ax = axes[1]
    entropies = [entropy(softmax_stable(z / t)) for t in tau_range]
    ax.plot(tau_range, entropies, 'b-', linewidth=2)
    ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Target H=1.0')
    ax.axhline(np.log(4), color='green', linestyle='--', alpha=0.5, label=f'Max H=ln(4)={np.log(4):.2f}')
    ax.axvline(tau_star, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Temperature τ')
    ax.set_ylabel('Entropy (nats)')
    ax.set_title('Entropy vs Temperature')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---
## Exercise 4: Top-p (Nucleus) Sampling ⭐⭐

**Problem:** Given probabilities after softmax:

| Token | cat | dog | fish | bird | rock | the | a | is |
|-------|-----|-----|------|------|------|-----|---|-----|
| P | 0.35 | 0.25 | 0.15 | 0.10 | 0.05 | 0.04 | 0.03 | 0.03 |

**Tasks:**
1. Sort tokens by probability (descending)
2. Compute cumulative sums
3. Find the nucleus for $p = 0.9$ (smallest set with cumulative ≥ 0.9)
4. Find the nucleus for $p = 0.5$
5. Renormalise the nucleus probabilities so they sum to 1
6. Compare: how many tokens in nucleus for $p \in \{0.5, 0.7, 0.9, 0.95, 1.0\}$?
7. Why is top-p better than top-k for varying entropy distributions?

In [ ]:
# === Exercise 4: Scaffold ===

tokens = ["cat", "dog", "fish", "bird", "rock", "the", "a", "is"]
probs  = np.array([0.35, 0.25, 0.15, 0.10, 0.05, 0.04, 0.03, 0.03])

# Task 1: Sort descending
# TODO: sorted_idx = ...

# Task 2: Cumulative sums
# TODO: cumsum = ...

# Task 3: Nucleus for p=0.9
# TODO: find cutoff index

# Task 4: Nucleus for p=0.5
# TODO

# Task 5: Renormalise
# TODO: nucleus_probs / nucleus_probs.sum()

# Task 6: Nucleus size for different p values
p_values = [0.5, 0.7, 0.9, 0.95, 1.0]
# TODO: for each p, find nucleus size

In [ ]:
# === Exercise 4: Solution ===

tokens = ["cat", "dog", "fish", "bird", "rock", "the", "a", "is"]
probs  = np.array([0.35, 0.25, 0.15, 0.10, 0.05, 0.04, 0.03, 0.03])

# Task 1: Sort descending (already sorted, but let's be explicit)
sorted_idx = np.argsort(probs)[::-1]
sorted_tokens = [tokens[i] for i in sorted_idx]
sorted_probs = probs[sorted_idx]
print("=== Step 1: Sort by probability ===")
print(f"{'Rank':<6} {'Token':<8} {'P':<8}")
for rank, (t, p) in enumerate(zip(sorted_tokens, sorted_probs), 1):
    print(f"{rank:<6} {t:<8} {p:.2f}")

# Task 2: Cumulative sums
cumsum = np.cumsum(sorted_probs)
print(f"\n=== Step 2: Cumulative sums ===")
print(f"{'Token':<8} {'P':<8} {'CumSum':<10}")
print("-" * 26)
for t, p, cs in zip(sorted_tokens, sorted_probs, cumsum):
    print(f"{t:<8} {p:.2f}{'':<3} {cs:.2f}")

# Task 3: Nucleus for p=0.9
p_threshold = 0.9
cutoff_90 = np.searchsorted(cumsum, p_threshold) + 1
nucleus_90 = sorted_tokens[:cutoff_90]
nucleus_probs_90 = sorted_probs[:cutoff_90]
print(f"\n=== Step 3: Nucleus for p=0.9 ===")
print(f"Smallest set with cumsum ≥ 0.9: {nucleus_90} (k={cutoff_90})")
print(f"Cumsum at cutoff: {cumsum[cutoff_90-1]:.2f}")

# Task 4: Nucleus for p=0.5
cutoff_50 = np.searchsorted(cumsum, 0.5) + 1
nucleus_50 = sorted_tokens[:cutoff_50]
print(f"\n=== Step 4: Nucleus for p=0.5 ===")
print(f"Nucleus: {nucleus_50} (k={cutoff_50})")

# Task 5: Renormalise
renorm_90 = nucleus_probs_90 / nucleus_probs_90.sum()
print(f"\n=== Step 5: Renormalised nucleus (p=0.9) ===")
print(f"{'Token':<8} {'Original':<10} {'Renormalised':<14}")
print("-" * 34)
for t, p_orig, p_new in zip(nucleus_90, nucleus_probs_90, renorm_90):
    print(f"{t:<8} {p_orig:<10.4f} {p_new:<14.4f}")
print(f"Sum: {renorm_90.sum():.4f} ✓")

# Task 6: Nucleus size vs p
print(f"\n=== Step 6: Nucleus size comparison ===")
p_values = [0.5, 0.7, 0.9, 0.95, 1.0]
print(f"{'p':<6} {'k (nucleus size)':<18} {'Tokens in nucleus'}")
print("-" * 55)
for p_val in p_values:
    k = np.searchsorted(cumsum, p_val - 1e-9) + 1
    k = min(k, len(sorted_tokens))
    nuc = sorted_tokens[:k]
    print(f"{p_val:<6.2f} {k:<18} {nuc}")

# Task 7: Why top-p > top-k
print(f"\n=== Step 7: Why top-p adapts better than top-k ===")
print("top-k always keeps exactly k tokens regardless of distribution shape.")
print("top-p ADAPTS: peaked distribution → small nucleus; flat → large nucleus.")
print("\nExample:")
p_peaked = np.array([0.90, 0.05, 0.03, 0.01, 0.005, 0.003, 0.001, 0.001])
p_flat   = np.array([0.15, 0.14, 0.13, 0.13, 0.12, 0.11, 0.11, 0.11])
for label, dist in [("Peaked", p_peaked), ("Flat", p_flat)]:
    cs = np.cumsum(np.sort(dist)[::-1])
    k90 = np.searchsorted(cs, 0.9) + 1
    print(f"  {label}: top-k=3 always keeps 3 tokens, but top-p=0.9 keeps {k90} tokens")

---
## Exercise 5: Cross-Entropy Gradient Derivation ⭐⭐⭐

**Problem:** Given logits $z = [2.0, 1.0, 0.5]$ and target class $k = 0$:

**Tasks:**
1. Compute $p = \text{softmax}(z)$
2. Compute cross-entropy loss $\mathcal{L} = -\log p_k$
3. Derive the gradient analytically: $\frac{\partial \mathcal{L}}{\partial z_i} = p_i - \mathbf{1}[i=k]$
4. Verify numerically with finite differences: $\frac{\partial \mathcal{L}}{\partial z_i} \approx \frac{\mathcal{L}(z_i+\epsilon) - \mathcal{L}(z_i-\epsilon)}{2\epsilon}$
5. Show that the gradient is always in $[-1, 1]$ and sums to zero
6. What happens to the gradient as the model becomes more confident (scales logits by 10)?

In [ ]:
# === Exercise 5: Scaffold ===

z = np.array([2.0, 1.0, 0.5])
k = 0  # target class

# Task 1: softmax
p = None  # TODO

# Task 2: Cross-entropy loss
L = None  # TODO: -log(p[k])

# Task 3: Analytical gradient
grad_analytical = None  # TODO: p - one_hot(k)

# Task 4: Numerical gradient (finite differences)
eps = 1e-5
grad_numerical = np.zeros(3)
# TODO: for each i, compute (L(z+eps*e_i) - L(z-eps*e_i)) / (2*eps)

# Task 5: Verify properties
# TODO: check gradient sum = 0, all values in [-1, 1]

# Task 6: High-confidence regime
# TODO: scale z by 10, recompute

In [ ]:
# === Exercise 5: Solution ===

z = np.array([2.0, 1.0, 0.5])
k = 0  # target class
V = len(z)

# Task 1: Softmax
p = softmax_stable(z)
print("=== Task 1: Softmax ===")
print(f"z = {z}")
print(f"p = softmax(z) = [{', '.join(f'{x:.6f}' for x in p)}]")
print(f"Sum = {p.sum():.6f} ✓")

# Task 2: Cross-entropy loss
L = -np.log(p[k])
print(f"\n=== Task 2: Cross-entropy loss ===")
print(f"L = -log(p[{k}]) = -log({p[k]:.6f}) = {L:.6f}")

# Task 3: Analytical gradient
print(f"\n=== Task 3: Analytical gradient ===")
print("Derivation:")
print("  L = -log(softmax(z)_k) = -z_k + log(Σ exp(z_j))")
print("  ∂L/∂z_i = -δ_{ik} + exp(z_i)/Σexp(z_j) = p_i - 1[i=k]")
one_hot = np.zeros(V)
one_hot[k] = 1.0
grad_analytical = p - one_hot
print(f"\n  grad_analytical = p - one_hot({k})")
for i in range(V):
    sign = "target" if i == k else ""
    print(f"  ∂L/∂z_{i} = {p[i]:.6f} - {one_hot[i]:.1f} = {grad_analytical[i]:+.6f}  {sign}")

# Task 4: Numerical verification
print(f"\n=== Task 4: Numerical gradient verification ===")
eps = 1e-5
grad_numerical = np.zeros(V)
for i in range(V):
    z_plus = z.copy(); z_plus[i] += eps
    z_minus = z.copy(); z_minus[i] -= eps
    L_plus = -np.log(softmax_stable(z_plus)[k])
    L_minus = -np.log(softmax_stable(z_minus)[k])
    grad_numerical[i] = (L_plus - L_minus) / (2 * eps)

print(f"{'i':<4} {'Analytical':<14} {'Numerical':<14} {'|Error|':<14}")
print("-" * 46)
for i in range(V):
    err = abs(grad_analytical[i] - grad_numerical[i])
    print(f"{i:<4} {grad_analytical[i]:<14.8f} {grad_numerical[i]:<14.8f} {err:<14.2e}")
print(f"Max error: {np.max(np.abs(grad_analytical - grad_numerical)):.2e} ✓")

# Task 5: Properties
print(f"\n=== Task 5: Gradient properties ===")
print(f"Sum of gradients: {grad_analytical.sum():.10f} ≈ 0 ✓")
print(f"Min gradient: {grad_analytical.min():.6f} ≥ -1 ✓")
print(f"Max gradient: {grad_analytical.max():.6f} ≤ 1 ✓")
print(f"  (target gradient is always negative: push logit UP)")
print(f"  (non-target gradients always positive: push logits DOWN)")

# Task 6: High-confidence regime
print(f"\n=== Task 6: High-confidence regime ===")
for scale in [1, 5, 10, 50]:
    z_scaled = z * scale
    p_scaled = softmax_stable(z_scaled)
    L_scaled = -np.log(p_scaled[k])
    grad_scaled = p_scaled - one_hot
    print(f"  scale={scale:>2}: p[0]={p_scaled[0]:.6f}, L={L_scaled:.6f}, "
          f"grad = [{', '.join(f'{g:+.6f}' for g in grad_scaled)}]")
print("\nInsight: as confidence → 1, gradient → 0 (vanishing gradient for correct predictions)")
print("This is why cross-entropy is efficient: it only updates when the model is wrong!")

---
## Exercise 6: Scaling Law Extrapolation ⭐⭐⭐

**Problem:** Using Kaplan scaling law $L(N) = (N_c / N)^{0.076}$ and the Chinchilla rule (20 tokens per parameter):

**Given reference points:**
- 7B model: loss = 1.85 nats, trained on 2T tokens
- 70B model: loss = 1.62 nats, trained on 2T tokens

**Tasks:**
1. Fit $N_c$ from the 7B reference point
2. Predict loss for 700B parameters
3. Using Chinchilla rule, how many tokens should 700B model see?
4. Compute the FLOPs for training each model ($C \approx 6ND$)
5. Compare tokens/parameter ratios: which models are over/under-trained?
6. Given a fixed budget of $10^{25}$ FLOPs, what's the optimal $N$ and $D$?

In [ ]:
# === Exercise 6: Scaffold ===

alpha = 0.076
N_7B, L_7B, D_7B = 7e9, 1.85, 2e12
N_70B, L_70B, D_70B = 70e9, 1.62, 2e12
chinchilla_ratio = 20

# Task 1: Fit Nc from 7B reference
Nc = None  # TODO: N * L^(1/alpha)

# Task 2: Predict 700B loss
N_700B = 700e9
L_700B = None  # TODO: (Nc/N)^alpha

# Task 3: Chinchilla optimal tokens for 700B
D_700B_optimal = None  # TODO

# Task 4: FLOPs for each (C ≈ 6ND)
# TODO

# Task 5: Tokens/parameter ratios
# TODO

# Task 6: Optimal allocation for C = 10^25
C_budget = 1e25
# TODO: solve for optimal N and D

In [ ]:
# === Exercise 6: Solution ===

alpha = 0.076
N_7B, L_7B, D_7B = 7e9, 1.85, 2e12
N_70B, L_70B, D_70B = 70e9, 1.62, 2e12
chinchilla_ratio = 20

# Task 1: Fit Nc
# L(N) = (Nc/N)^alpha → Nc = N * L^(1/alpha)
Nc = N_7B * (L_7B ** (1 / alpha))
print("=== Task 1: Fit Nc ===")
print(f"L(N) = (Nc/N)^{alpha}")
print(f"Nc = N × L^(1/α) = {N_7B:.0e} × {L_7B}^(1/{alpha})")
print(f"Nc = {Nc:.4e}")

# Verify with 7B
L_7B_pred = (Nc / N_7B) ** alpha
print(f"Verify: L(7B) = ({Nc:.2e}/{N_7B:.2e})^{alpha} = {L_7B_pred:.4f} ≈ {L_7B} ✓")

# Task 2: Predict 700B loss
N_700B = 700e9
L_700B = (Nc / N_700B) ** alpha
print(f"\n=== Task 2: Predict 700B loss ===")
print(f"L(700B) = ({Nc:.2e}/{N_700B:.2e})^{alpha} = {L_700B:.4f} nats")
print(f"PPL(700B) = exp({L_700B:.4f}) = {np.exp(L_700B):.2f}")

# Also predict what 70B should give
L_70B_pred = (Nc / N_70B) ** alpha
print(f"\nPredicted L(70B) = {L_70B_pred:.4f} vs actual {L_70B}")
print(f"  Gap = {abs(L_70B_pred - L_70B):.4f} (different training data explains gap)")

# Task 3: Chinchilla optimal tokens
D_700B_optimal = chinchilla_ratio * N_700B
print(f"\n=== Task 3: Chinchilla optimal tokens for 700B ===")
print(f"D_optimal = {chinchilla_ratio} × {N_700B:.0e} = {D_700B_optimal:.2e} = {D_700B_optimal/1e12:.1f}T tokens")

# Task 4: FLOPs
print(f"\n=== Task 4: FLOPs comparison (C ≈ 6ND) ===")
models = [
    ("7B", N_7B, D_7B, L_7B),
    ("70B", N_70B, D_70B, L_70B),
    ("700B (Chinchilla)", N_700B, D_700B_optimal, L_700B),
]
print(f"{'Model':<22} {'N':<10} {'D':<12} {'C (FLOPs)':<14} {'L (nats)'}")
print("-" * 70)
for name, N, D, L in models:
    C = 6 * N * D
    print(f"{name:<22} {N/1e9:.0f}B{'':<5} {D/1e12:.1f}T{'':<7} {C:.2e}{'':<5} {L:.4f}")

# Task 5: Tokens/parameter ratios
print(f"\n=== Task 5: Training data efficiency ===")
print(f"{'Model':<22} {'Tokens/param':<14} {'vs Chinchilla (20:1)'}")
print("-" * 56)
for name, N, D, _ in models:
    ratio = D / N
    status = "✓ optimal" if abs(ratio - 20) < 5 else ("UNDER-trained" if ratio < 20 else "OVER-trained")
    print(f"{name:<22} {ratio:<14.0f} {status}")

# LLaMA 3 comparison
print(f"\n  LLaMA 3 8B:  15T/8B  = {15e12/8e9:.0f}:1 tokens/param → massively over-trained")
print(f"  LLaMA 3 70B: 15T/70B = {15e12/70e9:.0f}:1 tokens/param → over-trained (inference-optimal)")

# Task 6: Optimal allocation for fixed budget
C_budget = 1e25
print(f"\n=== Task 6: Optimal allocation for C = {C_budget:.0e} FLOPs ===")
# C ≈ 6ND, D = 20N (Chinchilla) → C = 6N × 20N = 120N²
# N = sqrt(C / 120)
N_optimal = np.sqrt(C_budget / (6 * chinchilla_ratio))
D_optimal = chinchilla_ratio * N_optimal
L_optimal = (Nc / N_optimal) ** alpha

print(f"C = 6ND = 6N × 20N = 120N²")
print(f"N* = √(C/120) = √({C_budget:.0e}/120) = {N_optimal:.2e} = {N_optimal/1e9:.1f}B params")
print(f"D* = 20 × N* = {D_optimal:.2e} = {D_optimal/1e12:.1f}T tokens")
print(f"Predicted loss: L = {L_optimal:.4f} nats, PPL = {np.exp(L_optimal):.2f}")
print(f"\nVerify: 6ND = 6 × {N_optimal:.2e} × {D_optimal:.2e} = {6*N_optimal*D_optimal:.2e} ≈ {C_budget:.0e} ✓")

---
## Exercise 7: DPO Loss by Hand ⭐⭐⭐

**Problem:** Compute DPO loss for a preference pair with $\beta = 0.1$.

**Given:**
- Winner response $y_w$: helpful, accurate explanation
- Loser response $y_l$: vague, incorrect explanation

| Quantity | Value |
|----------|-------|
| $\log\pi_\theta(y_w \mid x)$ | -2.0 |
| $\log\pi_\theta(y_l \mid x)$ | -2.5 |
| $\log\pi_{\text{ref}}(y_w \mid x)$ | -2.8 |
| $\log\pi_{\text{ref}}(y_l \mid x)$ | -2.3 |

**Tasks:**
1. Compute implicit rewards: $r(y) = \beta \log[\pi_\theta(y)/\pi_{\text{ref}}(y)]$
2. Compute reward margin: $r(y_w) - r(y_l)$
3. Compute DPO loss: $\mathcal{L} = -\log\sigma(\text{margin})$
4. Compute gradients: which direction should $\pi_\theta$ move?
5. After one update with $\eta=0.5$, recompute loss. Did it decrease?
6. What happens if you swap winner and loser? Interpret.

In [ ]:
# === Exercise 7: Scaffold ===

beta = 0.1
logp_w_theta  = -2.0   # log π_θ(y_w | x)
logp_l_theta  = -2.5   # log π_θ(y_l | x)
logp_w_ref    = -2.8   # log π_ref(y_w | x)
logp_l_ref    = -2.3   # log π_ref(y_l | x)

# Task 1: Implicit rewards
r_w = None  # TODO: beta * (logp_w_theta - logp_w_ref)
r_l = None  # TODO: beta * (logp_l_theta - logp_l_ref)

# Task 2: Margin
margin = None  # TODO: r_w - r_l

# Task 3: DPO loss
loss = None  # TODO: -log(sigmoid(margin))

# Task 4: Gradients
# TODO: which direction should logp_w_theta and logp_l_theta move?

# Task 5: After update
# TODO

# Task 6: Swap winner/loser
# TODO

In [ ]:
# === Exercise 7: Solution ===

beta = 0.1
logp_w_theta  = -2.0
logp_l_theta  = -2.5
logp_w_ref    = -2.8
logp_l_ref    = -2.3

# Task 1: Implicit rewards
print("=== Task 1: Implicit rewards ===")
r_w = beta * (logp_w_theta - logp_w_ref)
r_l = beta * (logp_l_theta - logp_l_ref)
print(f"r(y_w) = β × [logπ_θ(y_w) - logπ_ref(y_w)]")
print(f"       = {beta} × [{logp_w_theta} - ({logp_w_ref})]")
print(f"       = {beta} × {logp_w_theta - logp_w_ref} = {r_w:.4f}")
print(f"r(y_l) = {beta} × [{logp_l_theta} - ({logp_l_ref})] = {beta} × {logp_l_theta - logp_l_ref} = {r_l:.4f}")
print(f"\nInterpretation: r_w > 0 means θ increased winner prob vs ref")
print(f"                r_l < 0 means θ decreased loser prob vs ref (good!)")

# Task 2: Margin
print(f"\n=== Task 2: Reward margin ===")
margin = r_w - r_l
print(f"margin = r(y_w) - r(y_l) = {r_w:.4f} - ({r_l:.4f}) = {margin:.4f}")
print(f"  Positive margin → model correctly prefers winner ✓")

# Task 3: DPO loss
print(f"\n=== Task 3: DPO loss ===")
sig_m = sigmoid(margin)
loss = -np.log(sig_m)
print(f"σ(margin) = σ({margin:.4f}) = {sig_m:.6f}")
print(f"L_DPO = -log(σ({margin:.4f})) = -log({sig_m:.6f}) = {loss:.6f}")
print(f"  (loss > 0 always; approaches 0 as margin → +∞)")

# Task 4: Gradients
print(f"\n=== Task 4: Gradients ===")
weight = sigmoid(-margin)  # how "wrong" the model is
grad_w = -beta * weight    # gradient for logπ(y_w)
grad_l = +beta * weight    # gradient for logπ(y_l)
print(f"σ(-margin) = {weight:.6f}  (gradient weight)")
print(f"∂L/∂logπ_θ(y_w) = -β × σ(-m) = {grad_w:.6f}  ← NEGATIVE: increase winner prob")
print(f"∂L/∂logπ_θ(y_l) = +β × σ(-m) = {grad_l:.6f}  ← POSITIVE: decrease loser prob")
print(f"\nDirection: push winner UP, push loser DOWN")

# Task 5: After update
print(f"\n=== Task 5: After one gradient step (η=0.5) ===")
lr = 0.5
logp_w_new = logp_w_theta - lr * grad_w
logp_l_new = logp_l_theta - lr * grad_l
r_w_new = beta * (logp_w_new - logp_w_ref)
r_l_new = beta * (logp_l_new - logp_l_ref)
margin_new = r_w_new - r_l_new
loss_new = -np.log(sigmoid(margin_new))
print(f"logπ_θ(y_w): {logp_w_theta:.4f} → {logp_w_new:.4f} (increased ✓)")
print(f"logπ_θ(y_l): {logp_l_theta:.4f} → {logp_l_new:.4f} (decreased ✓)")
print(f"Margin: {margin:.4f} → {margin_new:.4f} (increased ✓)")
print(f"Loss:   {loss:.6f} → {loss_new:.6f} (decreased ✓)")

# Task 6: Swap winner and loser
print(f"\n=== Task 6: Swap winner ↔ loser ===")
r_w_swap = beta * (logp_l_theta - logp_l_ref)  # old loser as "winner"
r_l_swap = beta * (logp_w_theta - logp_w_ref)  # old winner as "loser"
margin_swap = r_w_swap - r_l_swap
loss_swap = -np.log(sigmoid(margin_swap))
print(f"Swapped margin: {margin_swap:.4f} (was {margin:.4f})")
print(f"Swapped loss:   {loss_swap:.6f} (was {loss:.6f})")
print(f"\nHigher loss when labels are swapped → DPO is correctly using preference signal")
print(f"The model would need to REVERSE its preference direction to reduce this loss")

---
## Exercise 8: Calibration Check (ECE) ⭐⭐⭐

**Problem:** A model makes 100 predictions with the following confidence-accuracy profile:

| Confidence bucket | Count | Accuracy |
|-------------------|-------|----------|
| (0.5, 0.6] | 10 | 0.50 |
| (0.6, 0.7] | 15 | 0.53 |
| (0.7, 0.8] | 25 | 0.64 |
| (0.8, 0.9] | 30 | 0.72 |
| (0.9, 1.0] | 20 | 0.85 |

**Tasks:**
1. Compute the mean confidence for each bin (use bin midpoint)
2. Compute the calibration gap $|\text{acc} - \text{conf}|$ for each bin
3. Compute ECE = $\sum \frac{n_m}{N} \times |\text{acc}_m - \text{conf}_m|$
4. Is this model overconfident or underconfident? By how much?
5. Find the temperature $T^*$ that would minimise ECE (simulate)
6. Plot a reliability diagram (if matplotlib available)

In [ ]:
# === Exercise 8: Scaffold ===

N_total = 100
bins = [
    # (lo, hi, count, accuracy)
    (0.5, 0.6, 10, 0.50),
    (0.6, 0.7, 15, 0.53),
    (0.7, 0.8, 25, 0.64),
    (0.8, 0.9, 30, 0.72),
    (0.9, 1.0, 20, 0.85),
]

# Task 1: Mean confidence per bin (midpoint)
# TODO

# Task 2: Calibration gap for each bin
# TODO

# Task 3: ECE
ece = None  # TODO

# Task 4: Over/underconfident?
# TODO

# Task 5: Find T* (simulate)
# TODO

# Task 6: Reliability diagram
# TODO

In [ ]:
# === Exercise 8: Solution ===

N_total = 100
bins = [
    (0.5, 0.6, 10, 0.50),
    (0.6, 0.7, 15, 0.53),
    (0.7, 0.8, 25, 0.64),
    (0.8, 0.9, 30, 0.72),
    (0.9, 1.0, 20, 0.85),
]

# Task 1 & 2: Midpoint confidence and gaps
print("=== Task 1-2: Calibration Analysis ===\n")
print(f"{'Bin':<12} {'Count':<8} {'Conf (mid)':<12} {'Accuracy':<10} {'|Gap|':<8} {'Direction'}")
print("-" * 62)

ece = 0.0
weighted_bias = 0.0
gaps = []
midpoints = []
accs = []

for lo, hi, count, acc in bins:
    conf = (lo + hi) / 2  # midpoint as mean confidence
    gap = abs(acc - conf)
    direction = "overconfident" if conf > acc else "underconfident" if conf < acc else "perfect"
    ece += (count / N_total) * gap
    weighted_bias += (count / N_total) * (conf - acc)
    gaps.append(gap)
    midpoints.append(conf)
    accs.append(acc)
    print(f"({lo:.1f},{hi:.1f}]{'':<4} {count:<8} {conf:<12.2f} {acc:<10.2f} {gap:<8.2f} {direction}")

# Task 3: ECE
print(f"\n=== Task 3: ECE ===")
print(f"ECE = Σ (nₘ/N) × |accₘ - confₘ|")
terms = []
for (lo, hi, count, acc) in bins:
    conf = (lo + hi) / 2
    gap = abs(acc - conf)
    term = (count / N_total) * gap
    terms.append(f"({count}/{N_total})×{gap:.2f}")
print(f"    = {' + '.join(terms)}")
print(f"    = {ece:.4f}")
print(f"\nECE = {ece:.4f} → {ece*100:.2f}% average calibration error")

# Task 4: Over/underconfident
print(f"\n=== Task 4: Overall calibration bias ===")
print(f"Weighted mean confidence: {sum((lo+hi)/2 * count/N_total for lo,hi,count,acc in bins):.4f}")
print(f"Weighted mean accuracy:  {sum(acc * count/N_total for _,_,count,acc in bins):.4f}")
print(f"Mean bias (conf - acc):  {weighted_bias:.4f}")
print(f"→ Model is OVERCONFIDENT by {weighted_bias:.4f} ({weighted_bias*100:.1f}%)")
print("  Every bin has confidence > accuracy")

# Task 5: Temperature scaling simulation
print(f"\n=== Task 5: Temperature scaling ===")
# Simulate: create synthetic logits matching the bin profile
np.random.seed(42)
all_confs = []
all_correct = []

for lo, hi, count, acc in bins:
    mid = (lo + hi) / 2
    # Generate logits such that max softmax ≈ mid
    # For binary: logit ≈ log(mid/(1-mid))
    for _ in range(count):
        logit_main = np.log(mid / (1 - mid + 1e-10))
        logits = np.array([logit_main, 0.0])  # binary
        p = softmax_stable(logits)
        all_confs.append(p[0])
        all_correct.append(1 if np.random.random() < acc else 0)

all_confs = np.array(all_confs)
all_correct = np.array(all_correct)

# Search for optimal temperature
best_T, best_ece = 1.0, ece
print(f"{'T':<8} {'ECE':<10} {'Mean conf':<12} {'Status'}")
print("-" * 42)
for T in [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0]:
    # Re-scale confidences
    logits_all = np.log(all_confs / (1 - all_confs + 1e-10))
    scaled_confs = 1 / (1 + np.exp(-logits_all / T))
    
    # Compute ECE
    bin_boundaries = np.linspace(0, 1, 11)
    ece_t = 0.0
    for i in range(10):
        mask = (scaled_confs > bin_boundaries[i]) & (scaled_confs <= bin_boundaries[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = all_correct[mask].mean()
        bin_conf = scaled_confs[mask].mean()
        ece_t += (mask.sum() / N_total) * abs(bin_acc - bin_conf)
    
    marker = " ← best" if ece_t < best_ece else ""
    if ece_t < best_ece:
        best_ece = ece_t
        best_T = T
    print(f"{T:<8.1f} {ece_t:<10.4f} {scaled_confs.mean():<12.4f} {marker}")

print(f"\nOptimal T* ≈ {best_T:.1f} (ECE drops from {ece:.4f} to {best_ece:.4f})")
print(f"  Higher T → softer probabilities → reduces overconfidence")

# Task 6: Reliability diagram
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    # Before calibration
    ax = axes[0]
    ax.bar(midpoints, accs, width=0.08, alpha=0.7, color='steelblue', edgecolor='black', label='Accuracy')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect calibration')
    for m, a, g in zip(midpoints, accs, gaps):
        ax.annotate(f'gap={g:.2f}', (m, a), fontsize=8, xytext=(5, 5), textcoords='offset points')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Reliability Diagram (ECE={ece:.4f})')
    ax.set_xlim(0.4, 1.05)
    ax.set_ylim(0.4, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # ECE components
    ax = axes[1]
    weights = [count/N_total for _,_,count,_ in bins]
    weighted_gaps = [w * g for w, g in zip(weights, gaps)]
    ax.bar(midpoints, weighted_gaps, width=0.08, color='coral', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Confidence bin')
    ax.set_ylabel('(n/N) × |acc - conf|')
    ax.set_title('ECE Contributions per Bin')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("\n=== Key insight ===")
print("ECE measures average miscalibration weighted by bin population.")
print("High-count bins (0.8-0.9: 30 samples) contribute most to ECE.")
print("Temperature scaling T>1 softens predictions to fix overconfidence.")